# Lab 2 — Extracting Stock Data Using Web Scraping

This notebook demonstrates how to extract historical stock data from web pages using the `requests` and `BeautifulSoup` libraries. We scrape structured HTML tables and load them into Pandas DataFrames.

**Stocks covered:** Netflix (NFLX) — example | Amazon (AMZN) — exercise

## Table of Contents
1. [Setup & Imports](#setup)
2. [Scraping Netflix Stock Data — BeautifulSoup](#netflix)
3. [Scraping with `pd.read_html`](#pandas-read)
4. [Exercise — Amazon Stock Data](#exercise)


In [1]:
# Install required libraries (run once)
!pip install pandas requests bs4 html5lib lxml plotly --quiet

In [2]:
import warnings
import pandas as pd
import requests
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Scraping Netflix Stock Data <a id='netflix'></a>

We extract a historical price table from an HTML page using `BeautifulSoup`. The target columns are: **Date, Open, High, Low, Close, Volume**.

### Step 1 — Send HTTP Request

In [3]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/netflix_data_webpage.html"
data = requests.get(url).text

### Step 2 — Parse HTML with BeautifulSoup

In [4]:
soup = BeautifulSoup(data, 'html.parser')

### Step 3 — Extract Table Data

We locate the `<tbody>` tag, iterate through each row `<tr>`, and extract individual cell values `<td>`.

In [5]:
netflix_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"])

for row in soup.find("tbody").find_all('tr'):
    col = row.find_all("td")
    if len(col) < 7:
        continue
    netflix_data = pd.concat([netflix_data, pd.DataFrame({
        "Date":      [col[0].text],
        "Open":      [col[1].text],
        "High":      [col[2].text],
        "Low":       [col[3].text],
        "Close":     [col[4].text],
        "Adj Close": [col[5].text],
        "Volume":    [col[6].text]
    })], ignore_index=True)

netflix_data.head()

,Date,Open,High,Low,Close,Adj Close,Volume
0,"Jun 01, 2021",504.01,536.13,482.14,528.21,528.21,"78,560,600"
1,"May 01, 2021",512.65,518.95,478.54,502.81,502.81,"66,927,600"
2,"Apr 01, 2021",529.93,563.56,499.00,513.47,513.47,"111,573,300"
3,"Mar 01, 2021",545.57,556.99,492.85,521.66,521.66,"90,183,900"
4,"Feb 01, 2021",536.79,566.65,518.28,538.85,538.85,"61,902,300"


## 2. Extracting Data with `pd.read_html` <a id='pandas-read'></a>

`pd.read_html()` is a convenient shortcut that parses all HTML tables on a page and returns them as a list of DataFrames. This is faster for well-structured pages but offers less control than manual scraping.

In [6]:
# Using the URL directly
netflix_dataframe = pd.read_html(url)[0]
netflix_dataframe.head()

,Date,Open,High,Low,Close*,Adj Close**,Volume
0,"Jun 01, 2021",504.01,536.13,482.14,528.21,528.21,78560600
1,"May 01, 2021",512.65,518.95,478.54,502.81,502.81,66927600
2,"Apr 01, 2021",529.93,563.56,499.00,513.47,513.47,111573300
3,"Mar 01, 2021",545.57,556.99,492.85,521.66,521.66,90183900
4,"Feb 01, 2021",536.79,566.65,518.28,538.85,538.85,61902300


## 3. Exercise — Amazon Stock Data <a id='exercise'></a>

We apply the same web scraping approach to extract historical stock data for **Amazon (AMZN)**.

In [7]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/amazon_data_webpage.html"
html_data = requests.get(url).text
soup = BeautifulSoup(html_data, "html.parser")

In [8]:
# Analysis 1: Page title
print("Page title:", soup.title.string)

Page title: Amazon.com, Inc. (AMZN) Stock Historical Prices & Data - Yahoo Finance


In [9]:
amazon_data = pd.DataFrame(columns=["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"])

for row in soup.find("tbody").find_all("tr"):
    col = row.find_all("td")
    if len(col) < 7:
        continue
    amazon_data = pd.concat([amazon_data, pd.DataFrame({
        "Date":      [col[0].text],
        "Open":      [col[1].text],
        "High":      [col[2].text],
        "Low":       [col[3].text],
        "Close":     [col[4].text],
        "Adj Close": [col[5].text],
        "Volume":    [col[6].text]
    })], ignore_index=True)

# Analysis 2: Column names
print("Columns:", list(amazon_data.columns))

Columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']


In [10]:
# First 5 rows
print(amazon_data.head())

# Analysis 3: Opening price of the last row
print("\nOpen price (last row):", amazon_data.tail(1)['Open'].values[0])

           Date      Open      High       Low     Close Adj Close       Volume
0  Jan 01, 2021  3,270.00  3,363.89  3,086.00  3,206.20  3,206.20   71,528,900
1  Dec 01, 2020  3,188.50  3,350.65  3,072.82  3,256.93  3,256.93   77,556,200
2  Nov 01, 2020  3,061.74  3,366.80  2,950.12  3,168.04  3,168.04   90,810,500
3  Oct 01, 2020  3,208.00  3,496.24  3,019.00  3,036.15  3,036.15  116,226,100
4  Sep 01, 2020  3,489.58  3,552.25  2,871.00  3,148.73  3,148.73  115,899,300

Open price (last row): 656.29
